# Lab 1: Vectorless RAG


## Setup

### Install Dependencies

In [20]:
!pip install -q pageindex openai python-dotenv pymupdf mermaid-py

### Import Libraries


In [21]:
import os       # for environment variables
import json     # for parsing LLM JSON responses
import pymupdf  # for extracting text from PDF files
from openai import OpenAI  # OpenAI-compatible client (works with OpenRouter)
from dotenv import load_dotenv  # loads API keys from .env file
from mermaid import Mermaid  # for rendering diagrams

### Load API Keys

In [22]:
load_dotenv("../.env")

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# If keys are missing, prompt the user to enter them
if not PAGEINDEX_API_KEY:
    PAGEINDEX_API_KEY = input("Enter your PageIndex API key (get one at https://pageindex.ai): ").strip()
if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = input("Enter your OpenRouter API key (get one at https://openrouter.ai): ").strip()

print("Keys loaded.")

Keys loaded.


### Set up the LLM

In [23]:
# Calls an LLM via OpenRouter's OpenAI-compatible API
# - prompt: the text to send to the model
# - model: which model to use (default is free tier)
def call_llm(prompt, model="nvidia/nemotron-3-ultra-550b-a55b:free"):
    # Create OpenAI-compatible client pointed at OpenRouter
    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)
    return client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],  # single user message
        temperature=0,   # deterministic output (no randomness)
        max_tokens=512  # cap response length
    ).choices[0].message.content.strip()

### Overall Workflow

In [ ]:
%%mermaidjs
flowchart TD
    A(["User asks a question"]) --> B["Submit PDF to PageIndex"]
    B --> C["PageIndex builds hierarchical tree<br/>(titles + summaries)"]
    C --> D["LLM finds relevant nodes from tree"]
    D --> E["Extract text from relevant pages"]
    E --> F["LLM generates answer from<br/>correct text"]
    F --> G(["Final answer"])

---
## Step 1 — Extract Text from PDF

In [24]:
PDF_PATH = "data/CCS 3.31.25 Earnings Release 8-K Exhibit 99.1.pdf"

doc = pymupdf.open(PDF_PATH)
# len(doc) = total pages; i goes from 0 to len(doc)-1
# i+1 makes page numbers 1-based (page 1, 2, 3...)
# doc.load_page(i).get_text() extracts raw text from page i
page_texts = {i+1: doc.load_page(i).get_text() for i in range(len(doc))}
doc.close()
print(f"Extracted text from {len(page_texts)} pages.")

Extracted text from 11 pages.


---
## Step 2 — Build Document Tree


### Build Document Tree


In [25]:
# PageIndex SDK for document tree generation and utilities
from pageindex import PageIndexClient
from pageindex import utils
import time

# Submit PDF to PageIndex for tree generation
pi = PageIndexClient(api_key=PAGEINDEX_API_KEY)
result = pi.submit_document(PDF_PATH)
doc_id = result["doc_id"]
print(f"Submitted: {doc_id}")

# Poll until processing completes (max 5 min)
elapsed = 0
while elapsed < 300:
    if pi.is_retrieval_ready(doc_id):
        break
    time.sleep(5)
    elapsed += 5
    print(f"  {elapsed}s...")
else:
    raise TimeoutError("PageIndex timeout")

# Retrieve the hierarchical tree (titles + summaries, no full text)
tree = pi.get_tree(doc_id, node_summary=True)["result"]
utils.print_tree(tree, exclude_fields=["text"])

Submitted: pi-cmrm9m8n900iz01o4iqososuo
  5s...
  10s...
  15s...
[{'title': 'Century Communities Reports First Quarte...',
  'node_id': '0000',
  'page_index': 1,
  'summary': 'Century Communities reported its Q1 2025...'},
 {'title': 'First Quarter 2025 Results',
  'node_id': '0001',
  'page_index': 1,
  'prefix_summary': 'The report details the Q1 2025 financial...',
  'nodes': [{'title': 'Balance Sheet and Liquidity',
             'node_id': '0002',
             'page_index': 2,
             'summary': 'In Q1 2025, the company maintained a str...'},
            {'title': 'Full Year 2025 Outlook',
             'node_id': '0003',
             'page_index': 2,
             'summary': '## Full Year 2025 Outlook\n\nScott Dixon, ...'},
            {'title': 'Webcast and Conference Call',
             'node_id': '0004',
             'page_index': 2,
             'summary': 'Century Communities will host a webcast ...'},
            {'title': 'About Century Communities',
             'node

---
## Step 3 — Ask a Question

In [26]:
QUERY = "What was the total revenue reported in the earnings release?"

---
## Step 4 — LLM Finds Relevant Sections

The LLM reads the tree (titles + summaries only — no full text) and picks which nodes likely contain the answer.

In [ ]:
%%mermaidjs
flowchart LR
    A[User Question] --> B[Full Tree]
    B --> C["Strip 'text' field<br/>utils.remove_fields()"]
    C --> D["Slim Tree<br/>(node_id + title + summary only)"]
    D --> E[LLM Analysis]
    A --> E
    E --> F["JSON Response<br/>{node_list: [...]}"]
    F --> G["Relevant Nodes<br/>identified by node_id"]

### Search the Tree


In [29]:
# Strip full text from tree — LLM only needs titles + summaries to pick relevant nodes
tree_slim = utils.remove_fields(tree.copy(), fields=["text"])

# Ask the LLM which nodes are relevant to the question
# We use JSON format so we can reliably extract structured data
search_prompt = f"""
IMPORTANT: You MUST respond with valid JSON only. No other text.

You are given a question and a document tree.
Each node has: node_id, title, summary.
Find all nodes likely to contain the answer.

Question: {QUERY}

Document tree:
{json.dumps(tree_slim, indent=2)}

Respond with ONLY this JSON format:
{{
    "thinking": "<your reasoning>",
    "node_list": ["node_id_1", "node_id_2"]
}}
"""

# Try to parse JSON — handle cases where LLM adds extra text or returns invalid JSON
try:
    result = json.loads(call_llm(search_prompt))
except json.JSONDecodeError:
    # Fallback: try to extract JSON from the response using regex
    import re
    match = re.search(r'\{{.*\}}', call_llm(search_prompt), re.DOTALL)
    if match:
        result = json.loads(match.group())
    else:
        print("Could not parse LLM response. Using empty result.")
        result = {"thinking": "", "node_list": []}


Reasoning: The question asks for the total revenue reported in the earnings release. Looking at the document tree, node 0000 (the main earnings release summary) states '$903.2 million in revenue'. Node 0001 (First Quarter 2025 Results) also states 'total revenues of $903.2M from 2,284 home deliveries'. Both nodes contain the answer to the question. 

Retrieved nodes:
  0000 | Pages 1 | Century Communities Reports First Quarter 2025 Results
  0001 | Pages 1-2 | First Quarter 2025 Results


### Map Nodes to Page Numbers

The LLM returns node IDs (e.g. `0000`, `0001`), but we need to know which **pages** those nodes correspond to. This step is required because the next cell extracts text from PDF pages — and it needs page numbers, not node IDs.

In [ ]:
# Map node IDs to their metadata (title, page range, etc.)
node_map = utils.create_node_mapping(tree, include_page_ranges=True, max_page=len(page_texts))

# Display the LLM's reasoning and which nodes it selected
print("\nReasoning:", result.get("thinking", ""), "\n")
print("Retrieved nodes:")
for nid in result.get("node_list", []):
    if nid in node_map:
        info = node_map[nid]
        # Show single page number or range (e.g. "3" or "3-5")
        pages = info['start_index'] if info['start_index'] == info['end_index'] else f"{info['start_index']}-{info['end_index']}"
        print(f"  {nid} | Pages {pages} | {info['node']['title']}")
    else:
        print(f"  {nid} | not found in tree")

---
## Step 5 — LLM Answers from Extracted Text

### Build Context

In [30]:
# Collect text from pages covered by retrieved nodes (deduplicating pages)
texts, seen = [], set()
for nid in result.get("node_list", []):
    if nid not in node_map:
        continue
    info = node_map[nid]
    # Loop through each page in the node's range
    for p in range(info["start_index"], info["end_index"] + 1):
        # Only add if we haven't seen this page and it has text
        if p not in seen and p in page_texts:
            texts.append(f"--- Page {p} ---\n{page_texts[p]}")
            seen.add(p)
# Join all extracted text into one context string
context = "\n\n".join(texts)
print(f"Using {len(context.splitlines())} lines of text.")

Using 89 lines of text.


### Generate Answer


In [32]:
# Send the extracted text + question to the LLM for the final answer.
# The prompt tells the LLM to only use the provided context.
answer_prompt = f"""
Answer the question based on the provided text.

Context:
{context}

Question: {QUERY}

Rules:
- Answer only from the context
- If the answer isn't there, say so
"""

answer = call_llm(answer_prompt)
print(answer)

$903.2 million


---
## Try It Yourself

Change `QUERY` above and re-run from **Step 4**.